In [0]:
%sql
CREATE CATALOG IF NOT EXISTS Fashion_Warehouse; 
USE CATALOG Fashion_Warehouse;
CREATE SCHEMA IF NOT EXISTS Bronze;
USE SCHEMA Bronze;

In [0]:
# Base path for Google Drive folder
base_url = "https://drive.google.com/drive/folders/1sYS3h_1_3nSgfGnPU6bAnku-RafSKsaE?usp=drive_link"

# Read the entire folder to get all file paths
df = (spark.read
    .format("csv")
    .option("databricks.connection", "googledriveconnection")
    .option("inferSchema", True)
    .option("header", True)
    .load(base_url))

# Get distinct file paths from the DataFrame
file_paths = df.select("_metadata.file_path").distinct().collect()

print(f"Found {len(file_paths)} CSV files:")
for idx, row in enumerate(file_paths, 1):
    file_id = row.file_path.split('/')[-1]
    print(f"{idx}. File ID: {file_id}")

Found 11 CSV files:
1. File ID: 1UhIxy5W26qTVohI5ZcCR3LEAl6h-ebqC
2. File ID: 1m77jj-aFbkaCx5b3IoZR_zaCMNfvVyAB
3. File ID: 15cXa4GNLb7DQ-8OrRYclq1M-f-KjbwUz
4. File ID: 1gg77W1Bhmba7YKt8WneJJeWDd7hgURpB
5. File ID: 1mV1C-0RuIzKXqrevHamL95ZOsITWzfaM
6. File ID: 1CoUpfSn6bqOU1LY8bfIAjpzPl2YmTgvd
7. File ID: 13bI2IfDErVKHP1zfVLkfeOI8qHcX7Lcw
8. File ID: 1jKg4VRysCbWX2JsPRZIAJA1Si-1fiDpz
9. File ID: 1EEafloM-IN0u6CSfxq9UxZ7paq9Jg2Re
10. File ID: 1NZNy_PlHt8SHBTfN5sMRLXHFuBYhtQge
11. File ID: 1DhMvdpUDI7Bbuji3mx7h7deQIDEG-vTT


In [0]:
# Base path for Google Drive folder
base_url = "https://drive.google.com/drive/folders/1sYS3h_1_3nSgfGnPU6bAnku-RafSKsaE?usp=drive_link"

# First, read the entire folder to get all file paths
df = (spark.read
    .format("csv")
    .option("databricks.connection", "googledriveconnection")
    .option("inferSchema", True)
    .option("header", True)
    .load(base_url))

# Get distinct file paths from the DataFrame
file_paths = df.select("_metadata.file_path").distinct().collect()

print(f"Found {len(file_paths)} CSV files:")
for idx, row in enumerate(file_paths, 1):
    file_id = row.file_path.split('/')[-1]
    print(f"{idx}. File ID: {file_id}")

# Function to clean column names (remove spaces and special characters)
def clean_column_names(df):
    for col_name in df.columns:
        # Replace spaces and special characters with underscores
        clean_name = col_name.strip().replace(' ', '_').replace(',', '').replace(';', '').replace('{', '').replace('}', '').replace('(', '').replace(')', '').replace('\n', '').replace('\t', '').replace('=', '')
        # Convert to lowercase for consistency
        clean_name = clean_name.lower()
        if col_name != clean_name:
            df = df.withColumnRenamed(col_name, clean_name)
    return df

# Loop through each file and save as a separate table
for idx, row in enumerate(file_paths, 1):
    file_url = row.file_path
    
    print(f"\nProcessing file {idx} of {len(file_paths)}...")
    print(f"File URL: {file_url}")
    
    # Read individual file
    df_single = (spark.read
        .format("csv")
        .option("databricks.connection", "googledriveconnection")
        .option("inferSchema", True)
        .option("header", True)
        .load(file_url))
    
    # Clean column names to remove invalid characters
    df_single = clean_column_names(df_single)
    
    row_count = df_single.count()
    
    # Create table name based on index
    table_name = f"fashion_warehouse.bronze.warehouse_data_{idx:02d}"
    
    # Save as separate table
    df_single.write.format("delta") \
        .mode("overwrite") \
        .saveAsTable(table_name)
    
    print(f"✓ Saved {row_count:,} rows to {table_name}")

print(f"\n✓ Complete! Created {len(file_paths)} separate tables in fashion_warehouse.bronze schema")

In [0]:
%sql
-- Rename tables to meaningful names based on their content
ALTER TABLE fashion_warehouse.bronze.warehouse_data_01 RENAME TO fashion_warehouse.bronze.scanner_events;
ALTER TABLE fashion_warehouse.bronze.warehouse_data_02 RENAME TO fashion_warehouse.bronze.stock_movements;
ALTER TABLE fashion_warehouse.bronze.warehouse_data_03 RENAME TO fashion_warehouse.bronze.orders;
ALTER TABLE fashion_warehouse.bronze.warehouse_data_04 RENAME TO fashion_warehouse.bronze.metadata;
ALTER TABLE fashion_warehouse.bronze.warehouse_data_05 RENAME TO fashion_warehouse.bronze.suppliers;
ALTER TABLE fashion_warehouse.bronze.warehouse_data_06 RENAME TO fashion_warehouse.bronze.customers;
ALTER TABLE fashion_warehouse.bronze.warehouse_data_07 RENAME TO fashion_warehouse.bronze.products;
ALTER TABLE fashion_warehouse.bronze.warehouse_data_08 RENAME TO fashion_warehouse.bronze.employees;
ALTER TABLE fashion_warehouse.bronze.warehouse_data_09 RENAME TO fashion_warehouse.bronze.deliveries;
ALTER TABLE fashion_warehouse.bronze.warehouse_data_10 RENAME TO fashion_warehouse.bronze.warehouses;

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS fashion_warehouse.silver;

In [0]:
%sql

CREATE TABLE IF NOT EXISTS fashion_warehouse.silver.warehouses (
    
    warehouse_code STRING NOT NULL,
    warehouse_name STRING NOT NULL,
    city STRING NOT NULL,
    capacity_units BIGINT NOT NULL,

    CONSTRAINT pk_warehouses
        PRIMARY KEY (warehouse_code)
)
USING DELTA;

In [0]:
%sql
INSERT INTO fashion_warehouse.silver.warehouses
(
    warehouse_code,
    warehouse_name,
    city,
    capacity_units
)
SELECT
    CAST(warehouse_code AS VARCHAR(10)),
    CAST(warehouse_name AS STRING),
    CAST(city AS STRING),
    CAST(capacity_units AS BIGINT)
FROM fashion_warehouse.bronze.warehouses;

In [0]:
%sql
SELECT * FROM fashion_warehouse.silver.warehouses
LIMIT 5; 

warehouse_code,warehouse_name,city,capacity_units
WH01,London Distribution Centre,London,1696205
WH02,Manchester Distribution Centre,Manchester,1526803
WH03,Birmingham Distribution Centre,Birmingham,526790
WH04,Nottingham Distribution Centre,Nottingham,906482
WH05,Leicester Distribution Centre,Leicester,1072316


In [0]:
%sql

DELETE FROM fashion_warehouse.silver.warehouses 
WHERE warehouse_code IN (
WITH DUPLICATE_WAREHOUSE AS (
SELECT
COUNT(warehouse_code) OVER (PARTITION BY warehouse_code ORDER BY WAREHOUSE_CODE) AS DUPLICATE
FROM fashion_warehouse.bronze.warehouses
)
SELECT * FROM DUPLICATE_WAREHOUSE
WHERE DUPLICATE > 1);

In [0]:
%sql
CREATE TABLE IF  NOT EXISTS fashion_warehouse.silver.suppliers 
(
    supplier_name STRING PRIMARY KEY NOT NULL,
    country STRING NOT NULL,
    supplier_tier STRING NOT NULL,
    Lead_time_days DOUBLE NOT NULL,
    active_flag INT NOT NULL
)
USING DELTA;

In [0]:
%sql
INSERT INTO fashion_warehouse.silver.suppliers 
    (supplier_name, 
    country, 
    supplier_tier, 
    Lead_time_days, 
    active_flag)
SELECT 
    CAST(supplier_name AS STRING),
    CAST(country AS STRING),
    CAST(supplier_tier AS STRING),
    CAST(Lead_time_days AS DOUBLE),
    CAST(active_flag AS INT)
FROM fashion_warehouse.bronze.suppliers


In [0]:
%sql
DELETE FROM fashion_warehouse.silver.suppliers
WHERE supplier_name IN (
    WITH DUPLICATE  AS (
    SELECT 
    ROW_NUMBER() OVER(PARTITION BY supplier_name ORDER BY supplier_name) AS DUPLICATE 
    FROM fashion_warehouse.silver.suppliers
            ) 
            SELECT * FROM DUPLICATE
            WHERE DUPLICATE > 1);

In [0]:
%sql
SELECT * FROM fashion_warehouse.silver.suppliers
LIMIT 5; 

supplier_name,country,supplier_tier,Lead_time_days,active_flag
Supplier 0001,India,B,20.0,1
Supplier 0002,India,B,29.0,1
Supplier 0003,Turkey,A,23.0,1
Supplier 0004,Turkey,B,8.0,1
Supplier 0005,Portugal,C,8.0,1


In [0]:
%sql

CREATE TABLE IF NOT EXISTS fashion_warehouse.silver.products
(
    sku STRING PRIMARY KEY NOT NULL,
    product STRING NOT NULL,
    brand STRING NOT NULL,
    category STRING NOT NULL,
    size STRING NOT NULL,
    colour STRING NOT NULL,
    unit_cost DOUBLE NOT NULL,
    unit_price DOUBLE NOT NULL,
    supplier_name STRING NOT NULL,
    active_flag INT NOT NULL,

    CONSTRAINT fk_supplier
        FOREIGN KEY (supplier_name)
        REFERENCES fashion_warehouse.silver.suppliers (supplier_name)
);




In [0]:

%sql
-- Joined table to get suppliers name instead of generic supplier ids 
-- Removed duplicates SKUs 

INSERT INTO fashion_warehouse.silver.products (
    sku, 
    product, 
    brand, 
    category, 
    size, 
    colour, 
    unit_cost, 
    unit_price, 
    supplier_name,
    active_flag
)
    SELECT 
        sku, 
        product, 
        brand, 
        category, 
        size, 
        colour, 
        unit_cost, 
        unit_price, 
        supplier_name,
        active_flag
    FROM (
            SELECT 
                CAST(p.sku AS STRING),
                CAST(REGEXP_REPLACE(p.product_name, '\\s+\\d+$', '') AS STRING) AS Product,
                CAST(p.brand AS STRING),
                CAST(p.category AS STRING),
                CAST(p.size AS STRING),
                CAST(p.colour AS STRING),
                CAST(p.unit_cost AS DOUBLE),
                CAST(p.unit_price AS DOUBLE),
                CAST(s.supplier_name AS STRING) AS supplier_name,
                CAST(p.active_flag AS STRING),
                ROW_NUMBER() OVER (PARTITION BY p.sku ORDER BY p.sku) AS RNK 
            FROM fashion_warehouse.bronze.products p
JOIN fashion_warehouse.bronze.suppliers s
ON p.supplier_id = s.supplier_id)
    WHERE RNK = 1;

In [0]:
%sql
SELECT * FROM fashion_warehouse.silver.products
LIMIT 5;

sku,product,brand,category,size,colour,unit_cost,unit_price,supplier_name,active_flag
SKU-000001,Levi's Sportswear,Levi's,Sportswear,XL,Red,87.01,100.06,Supplier 0039,1
SKU-000002,Vans Womenswear,Vans,Womenswear,L,Blue,104.09,133.57,Supplier 0272,1
SKU-000003,Vans Footwear,Vans,Footwear,L,White,67.69,92.32,Supplier 0462,1
SKU-000004,Reebok Footwear,Reebok,Footwear,XL,Blue,71.6,97.42,Supplier 0382,1
SKU-000005,Puma Womenswear,Puma,Womenswear,XXL,Blue,69.54,128.68,Supplier 0135,1


In [0]:
%sql
CREATE TABLE fashion_warehouse.silver.employees
(
 employee_id INT PRIMARY KEY,
 first_name STRING,
 last_name STRING,
 department STRING,
 job_title STRING,
 warehouse_id STRING,
 hire_date DATE,
 status STRING
);


In [0]:
%sql
INSERT INTO fashion_warehouse.silver.employees
(
  employee_id, 
  first_name,
  last_name,
  department,
  job_title,
  warehouse_id,
  hire_date,
  status
)
SELECT 
    employee_id, 
    first_name,
    last_name,
    department,
    job_title,
    warehouse_id,
    hire_date,
    status
FROM (
        SELECT 
        CAST(employee_id AS INT), 
        CAST(first_name AS STRING),
        CAST(last_name AS STRING),
        CAST(department AS STRING),
        CAST(job_title AS STRING),
        CAST(warehouse_id AS STRING),
        CAST(hire_date AS DATE),
        CAST(status AS STRING),
            ROW_NUMBER() OVER (PARTITION BY employee_id ORDER BY employee_id) AS duplicate
        FROM fashion_warehouse.bronze.employees)
WHERE duplicate = 1; 


In [0]:
%sql
SELECT * FROM fashion_warehouse.silver.employees
LIMIT 5;

employee_id,first_name,last_name,department,job_title,warehouse_id,hire_date,status
1001,Joseph,Martin,Inventory,Inventory Controller,4,2023-02-23,Active
1002,Sarah,Evans,Inventory,Inventory Controller,20,2022-04-25,Active
1003,Amelia,Williams,Packing,Packer,13,2024-12-27,Active
1004,Michael,Wright,Inventory,Inventory Controller,2,2024-10-03,Active
1005,David,Clarke,Dispatch,Dispatch Operative,11,2022-01-19,Active


In [0]:
%sql
CREATE TABLE IF NOT EXISTS fashion_warehouse.silver.customers 
(
customer_id INT PRIMARY KEY,
first_name STRING,
last_name STRING,
email STRING,
city STRING,
created_date TIMESTAMP,
customer_type STRING
)
USING DELTA;

In [0]:
%sql
-- Joined orders to determine returning customers.
-- Removed Duplicates 

INSERT INTO TABLE fashion_warehouse.silver.customers (
    customer_id,
    first_name,
    last_name,
    email,
    city,
    created_date,
    customer_type
)
    SELECT
    customer_id,
        first_name,
        last_name,
        email,
        city,
        created_date,
        customer_type
    FROM (
            WITH customers_order AS (
                SELECT 
                c.customer_id,
                c.first_name,
                c.last_name,
                CONCAT(c.first_name,'.',c.last_name,'@gmail.com') AS email,
                c.city,
                c.created_date
                FROM fashion_warehouse.bronze.customers c
            ), 
            returning_new AS (
                SELECT
                o.customer_id,
                count(o.customer_id) as orders,
                CASE WHEN count(o.customer_id) < 2 THEN 'New' ELSE 'Returning' END as customer_type
            FROM fashion_warehouse.bronze.orders o
            Group by customer_id
            ORDER BY customer_id
)
    SELECT 
            co.customer_id,
            co.first_name,
            co.last_name,
            CONCAT(co.first_name,'.',co.last_name,'@gmail.com') AS email,
            co.city,
            co.created_date,
            rn.customer_type,
            ROW_NUMBER() OVER (PARTITION BY co.customer_id ORDER BY co.customer_id) AS duplicate
    FROM customers_order co
            INNER JOIN returning_new rn
            ON co.customer_id = rn.customer_id
            order by co.customer_id)
    WHERE duplicate = 1;

In [0]:
%sql
SELECT * FROM fashion_warehouse.silver.customers
LIMIT 5; 

customer_id,first_name,last_name,email,city,created_date,customer_type
100001,David,Davies,David.Davies@gmail.com,Sheffield,2026-01-26T00:00:00.000Z,Returning
100002,Charlotte,White,Charlotte.White@gmail.com,Leicester,2026-02-10T00:00:00.000Z,Returning
100003,Joseph,Young,Joseph.Young@gmail.com,Bristol,2026-05-25T00:00:00.000Z,Returning
100004,William,Harris,William.Harris@gmail.com,Leicester,2026-06-24T00:00:00.000Z,Returning
100005,Grace,Jones,Grace.Jones@gmail.com,Sheffield,2025-11-22T00:00:00.000Z,Returning


In [0]:
%sql
CREATE TABLE IF NOT EXISTS fashion_warehouse.silver.orders(
    order_id INT PRIMARY KEY,
    order_date TIMESTAMP,
    customer_id INT,
    warehouse_code VARCHAR(10),
    sku STRING,
    quantity INT,
    sales_channel STRING,
    order_status STRING,
    unit_price DOUBLE,
    order_value DOUBLE,
CONSTRAINT fk_customer FOREIGN KEY (customer_id) REFERENCES fashion_warehouse.silver.customers (customer_id),
CONSTRAINT fk_warehouses FOREIGN KEY (warehouse_code) REFERENCES fashion_warehouse.silver.warehouses (warehouse_code),
CONSTRAINT fk_product FOREIGN KEY (sku) REFERENCES fashion_warehouse.silver.products (sku))
USING DELTA;

In [0]:
%sql
INSERT INTO Fashion_warehouse.silver.orders(
    order_id,
    order_date,
    customer_id,
    warehouse_code,
    sku,
    quantity,
    sales_channel,
    order_status,
    unit_price,
    order_value
)
SELECT 
    order_id,
    order_date,
    customer_id,
    warehouse_code,
    sku,
    quantity,
    sales_channel,
    order_status,
    unit_price,
    order_value
FROM (
SELECT
        o.order_id,
        o.order_date,
        c.customer_id,
        ws.warehouse_code,
        ps.sku,
        o.quantity,
        o.sales_channel,
        o.order_status,
        o.unit_price,
        ROUND(o.quantity * o.unit_price,2) AS order_value,
        ROW_NUMBER()  OVER (PARTITION BY  o.order_id ORDER BY  o.order_id) AS duplicate
FROM fashion_warehouse.bronze.orders o   
LEFT JOIN fashion_warehouse.silver.customers c
ON o.customer_id = c.customer_id
LEFT JOIN fashion_warehouse.bronze.warehouses bw
ON o.warehouse_id = bw.warehouse_id
LEFT JOIN fashion_warehouse.silver.warehouses ws
ON bw.warehouse_code = ws.warehouse_code
LEFT JOIN fashion_warehouse.bronze.products bp
ON o.product_id = bp.product_id
LEFT JOIN fashion_warehouse.silver.products ps
ON bp.sku = ps.sku)
WHERE duplicate =1;

In [0]:
%sql
SELECT * FROM Fashion_warehouse.silver.orders
LIMIT 5;

order_id,order_date,customer_id,warehouse_code,sku,quantity,sales_channel,order_status,unit_price,order_value
1,2025-10-03T16:08:32.000Z,132555,WH12,SKU-000471,4,App,Completed,130.77,523.08
2,2025-06-16T14:39:16.000Z,105360,WH03,SKU-002332,5,Store,Completed,233.84,1169.2
3,2025-03-09T21:34:01.000Z,150392,WH15,SKU-001623,4,Store,Completed,69.22,276.88
4,2025-02-15T17:23:36.000Z,146907,WH03,SKU-004783,4,Web,Completed,142.25,569.0
5,2026-02-23T04:24:58.000Z,126838,WH02,SKU-004056,2,Web,Completed,84.31,168.62


In [0]:
%sql
CREATE TABLE IF NOT EXISTS fashion_warehouse.silver.deliveries
(
    delivery_id INT PRIMARY KEY,
    supplier_name STRING,
    warehouse_code STRING,
    expected_date DATE,
    actual_date DATE,
    units_expected INT,
    units_received INT,
    units_diff INT,
    delivery_status STRING,
    delay_days INT,


CONSTRAINT fk_warehouses FOREIGN KEY (warehouse_code) REFERENCES fashion_warehouse.silver.warehouses (warehouse_code),
CONSTRAINT fk_suppliers FOREIGN KEY (supplier_name) REFERENCES fashion_warehouse.silver.suppliers (supplier_name))

USING DELTA;

In [0]:
%sql
INSERT INTO fashion_warehouse.silver.deliveries (
    delivery_id, 
    supplier_name,
    warehouse_code,
    expected_date,
    actual_date,
    units_expected,
    units_received,
    units_diff,
    delivery_status,
    delay_days
)
SELECT 
delivery_id, 
    supplier_name,
    warehouse_code,
    expected_date,
    actual_date,
    units_expected,
    units_received,
    units_diff,
    delivery_status,
    delay_days
FROM (
        SELECT 
            CAST(delivery_id AS INT),
            CAST(supplier_name AS STRING),
            CAST(w.warehouse_code AS STRING),
            CAST(expected_date AS DATE),
            CAST(actual_date AS DATE),
            CAST(units_expected AS INT),
            CAST(units_received AS INT), 
            CAST(units_received AS INT) - CAST(units_expected AS INT) AS units_diff,
        CASE WHEN units_expected = units_received THEN 'Fully received' 
                WHEN units_expected > units_received THEN 'Partial Received'
                WHEN units_expected < units_received THEN 'Over Received'
                ELSE 'Something Wrong' END AS delivery_status,
    DATEDIFF(actual_date, expected_date) AS delay_days,
    ROW_NUMBER() OVER(PARTITION BY delivery_id ORDER BY delivery_id) AS duplicate 

FROM fashion_warehouse.bronze.deliveries d   
    LEFT JOIN fashion_warehouse.bronze.warehouses w 
    ON d.warehouse_id = w.warehouse_id
    LEFT JOIN fashion_warehouse.bronze.suppliers s
    ON d.supplier_id = s.supplier_id
    ORDER BY actual_date DESC)
WHERE duplicate = 1 
;

In [0]:
%sql
SELECT * FROM fashion_warehouse.silver.deliveries
LIMIT 5; 

delivery_id,supplier_name,warehouse_code,expected_date,actual_date,units_expected,units_received,units_diff,delivery_status,delay_days
27023,Supplier 0041,WH11,2026-09-04,2026-09-10,913,3930,3017,Over Received,6
31927,Supplier 0341,WH07,2026-09-04,2026-09-10,1953,4974,3021,Over Received,6
43488,Supplier 0109,WH20,2026-09-04,2026-09-10,652,1510,858,Over Received,6
2490,Supplier 0142,WH18,2026-09-04,2026-09-09,2294,1300,-994,Partial Received,5
48889,Supplier 0088,WH04,2026-09-04,2026-09-09,2397,601,-1796,Partial Received,5


In [0]:
%sql
CREATE TABLE IF NOT EXISTS fashion_warehouse.silver.scanner_events
(
    event_id INT PRIMARY KEY,
    timestamp TIMESTAMP,
    scanner_id STRING,
    warehouse_code STRING,
    warehouse_name STRING,
    department STRING,
    action STRING,
    sku STRING,
    quantity INT,
    bin_location STRING,
FOREIGN KEY (warehouse_code) REFERENCES fashion_warehouse.silver.warehouses (warehouse_code),   
FOREIGN KEY (sku) REFERENCES fashion_warehouse.silver.products (sku)
)
USING DELTA;


   

In [0]:
%sql
INSERT INTO fashion_warehouse.silver.scanner_events (
    event_id ,
    timestamp,
    scanner_id,
    warehouse_code,
    warehouse_name,
    department,
    action,
    sku,
    quantity,
    bin_location
)
SELECT 
    CAST(se.event_id AS INT) ,
    CAST(se.timestamp AS TIMESTAMP),
    CAST(se.scanner_id AS STRING),
    CAST(w.warehouse_code AS STRING),
    CAST(w.warehouse_name AS STRING),
    CAST(se.department AS STRING),
    CAST(se.action AS STRING),
    CAST(p.sku AS STRING),
    CAST(se.quantity AS INT),
    CAST(se.bin_location AS STRING)
FROM fashion_warehouse.bronze.scanner_events se  
LEFT JOIN fashion_warehouse.bronze.warehouses w
ON se.warehouse_id = w.warehouse_id
LEFT JOIN fashion_warehouse.bronze.products bp
ON se.product_id = bp.product_id
LEFT JOIN fashion_warehouse.silver.products p
ON bp.sku = p.sku;

  


In [0]:
%sql
SELECT * FROM fashion_warehouse.silver.scanner_events
LIMIT 5; 

event_id,timestamp,scanner_id,warehouse_code,warehouse_name,department,action,sku,quantity,bin_location
1,2025-07-04T16:47:54.000Z,SC-018,WH20,Glasgow Distribution Centre,Packing,PACK,SKU-001592,45,D-42-02
2,2025-05-27T10:43:02.000Z,SC-485,WH14,Nottingham Distribution Centre,Picking,PACK,SKU-000241,19,E-47-01
3,2026-07-06T08:37:09.000Z,SC-071,WH03,Birmingham Distribution Centre,Packing,DISPATCH,SKU-002307,22,E-15-04
4,2025-11-01T07:02:31.000Z,SC-213,WH18,Liverpool Distribution Centre,Returns,PICK,SKU-001767,4,G-07-02
5,2025-05-03T02:53:57.000Z,SC-470,WH12,Manchester Distribution Centre,Packing,PICK,SKU-000760,43,G-53-05


In [0]:
%sql
CREATE TABLE IF NOT EXISTS fashion_warehouse.silver.stock_movements
(
    movement_id INT PRIMARY KEY,
    timestamp TIMESTAMP,
    warehouse_code STRING,
    sku STRING,
    product_name STRING,
    employee_id INT,
    movement_type STRING,
    quantity INT,
    reference_id INT,
    
    CONSTRAINT fk_stock_warehouse FOREIGN KEY (warehouse_code) REFERENCES fashion_warehouse.silver.warehouses (warehouse_code),
    CONSTRAINT fk_stock_product FOREIGN KEY (sku) REFERENCES fashion_warehouse.silver.products (sku),
    CONSTRAINT fk_employees FOREIGN KEY (employee_id) REFERENCES fashion_warehouse.silver.employees (employee_id)

)
USING DELTA;

In [0]:
%sql
INSERT INTO fashion_warehouse.silver.stock_movements (
    movement_id,
    timestamp,
    warehouse_code,
    sku,
    product_name,
    employee_id,
    movement_type,
    quantity,
    reference_id
)
SELECT 
    movement_id,
    timestamp,
    warehouse_code,
    sku,
    product_name,
    employee_id,
    movement_type,
    quantity,
    reference_id
FROM (
    SELECT 
        CAST(s.movement_id AS INT),
        CAST(s.timestamp AS TIMESTAMP),
        CAST(w.warehouse_code AS STRING),
        CAST(p.sku AS STRING),
        CAST(p.product AS STRING) AS product_name,
        CAST(COALESCE(s.employee_id, e.employee_id) AS INT) AS employee_id,
        CAST(s.movement_type AS STRING),
        CAST(s.quantity AS INT),
        CAST(s.reference_id AS INT),
        ROW_NUMBER() OVER (PARTITION BY s.movement_id ORDER BY s.movement_id) AS duplicate
    FROM fashion_warehouse.bronze.stock_movements s
    LEFT JOIN fashion_warehouse.bronze.scanner_events e
        ON s.movement_id = e.event_id
    LEFT JOIN fashion_warehouse.bronze.warehouses bw
        ON s.warehouse_id = bw.warehouse_id
    LEFT JOIN fashion_warehouse.silver.warehouses w
        ON bw.warehouse_code = w.warehouse_code
    LEFT JOIN fashion_warehouse.bronze.products bp
        ON s.product_id = bp.product_id
    LEFT JOIN fashion_warehouse.silver.products p
        ON bp.sku = p.sku
)
WHERE duplicate = 1;

In [0]:
%sql
SELECT * FROM fashion_warehouse.silver.stock_movements
LIMIT 5; 

movement_id,timestamp,warehouse_code,sku,product_name,employee_id,movement_type,quantity,reference_id
1,2025-09-26T14:49:44.000Z,WH12,SKU-002633,Puma Footwear,1616,ADJUSTMENT,97,62374
2,2025-10-08T17:18:12.000Z,WH13,SKU-004020,Puma Accessories,1673,TRANSFER,29,71277
3,2025-07-07T22:47:58.000Z,WH08,SKU-000814,Under Armour Accessories,2679,ADJUSTMENT,27,96692
4,2026-02-22T16:36:15.000Z,WH04,SKU-002475,Reebok Womenswear,2957,ADJUSTMENT,35,2736
5,2025-02-07T06:44:15.000Z,WH16,SKU-001763,Under Armour Accessories,1840,RETURN,37,36083


In [0]:
%sql
CREATE OR REPLACE TABLE fashion_warehouse.gold.dim_products 
(
    sku STRING,
    product STRING,
    brand STRING,
    category STRING,
    size STRING,
    colour STRING,
    unit_cost DOUBLE,
    unit_price DOUBLE,
    supplier_name STRING,
    active_flag STRING,
    CONSTRAINT pk_products PRIMARY KEY (sku),
    CONSTRAINT fk_supplier FOREIGN KEY (supplier_name) REFERENCES fashion_warehouse.silver.suppliers (supplier_name)
)
USING DELTA;

In [0]:
%sql
INSERT INTO fashion_warehouse.gold.dim_products (
    sku,
    product,
    brand,
    category,
    size,
    colour,
    unit_cost,
    unit_price,
    supplier_name,
    active_flag
)
SELECT 
    sku,
    product,
    brand,
    category,
    size,
    colour,
    unit_cost,
    unit_price,
    supplier_name,
    CAST(active_flag AS STRING)
FROM fashion_warehouse.silver.products;

In [0]:
%sql
SELECT * FROM fashion_warehouse.gold.dim_products 
LIMIT 5; 

sku,product,brand,category,size,colour,unit_cost,unit_price,supplier_name,active_flag
SKU-000001,Levi's Sportswear,Levi's,Sportswear,XL,Red,87.01,100.06,Supplier 0039,1
SKU-000002,Vans Womenswear,Vans,Womenswear,L,Blue,104.09,133.57,Supplier 0272,1
SKU-000003,Vans Footwear,Vans,Footwear,L,White,67.69,92.32,Supplier 0462,1
SKU-000004,Reebok Footwear,Reebok,Footwear,XL,Blue,71.6,97.42,Supplier 0382,1
SKU-000005,Puma Womenswear,Puma,Womenswear,XXL,Blue,69.54,128.68,Supplier 0135,1


In [0]:
%sql
CREATE OR REPLACE TABLE fashion_warehouse.gold.dim_customers
(
    customer_id INT,
    first_name STRING,
    last_name STRING,
    full_name STRING,
    email STRING,
    city STRING,
    created_date TIMESTAMP,
    customer_type STRING,
    CONSTRAINT pk_customers PRIMARY KEY (customer_id)
)
USING DELTA;

In [0]:
%sql
INSERT INTO fashion_warehouse.gold.dim_customers
SELECT 
    customer_id,
    first_name,
    last_name,
    CONCAT(first_name, ' ', last_name) AS full_name,
    email,
    city,
    created_date,
    customer_type
FROM fashion_warehouse.silver.customers;

In [0]:
%sql
SELECT * FROM fashion_warehouse.gold.dim_customers
LIMIT 5;  

customer_id,first_name,last_name,full_name,email,city,created_date,customer_type
100001,David,Davies,David Davies,David.Davies@gmail.com,Sheffield,2026-01-26T00:00:00.000Z,Returning
100002,Charlotte,White,Charlotte White,Charlotte.White@gmail.com,Leicester,2026-02-10T00:00:00.000Z,Returning
100003,Joseph,Young,Joseph Young,Joseph.Young@gmail.com,Bristol,2026-05-25T00:00:00.000Z,Returning
100004,William,Harris,William Harris,William.Harris@gmail.com,Leicester,2026-06-24T00:00:00.000Z,Returning
100005,Grace,Jones,Grace Jones,Grace.Jones@gmail.com,Sheffield,2025-11-22T00:00:00.000Z,Returning


In [0]:
%sql
CREATE OR REPLACE TABLE fashion_warehouse.gold.dim_warehouses
(
    warehouse_code STRING,
    warehouse_name STRING,
    city STRING,
    capacity_units BIGINT,
    CONSTRAINT pk_warehouses PRIMARY KEY (warehouse_code)
)
USING DELTA;

In [0]:
%sql
INSERT INTO fashion_warehouse.gold.dim_warehouses
SELECT 
    warehouse_code,
    warehouse_name,
    city,
    capacity_units
FROM fashion_warehouse.silver.warehouses;

In [0]:
%sql
SELECT * FROM fashion_warehouse.gold.dim_warehouses
LIMIT 5; 

warehouse_code,warehouse_name,city,capacity_units
WH01,London Distribution Centre,London,1696205
WH02,Manchester Distribution Centre,Manchester,1526803
WH03,Birmingham Distribution Centre,Birmingham,526790
WH04,Nottingham Distribution Centre,Nottingham,906482
WH05,Leicester Distribution Centre,Leicester,1072316


In [0]:
%sql
CREATE OR REPLACE TABLE fashion_warehouse.gold.dim_employees
(
    employee_id INT,
    first_name STRING,
    last_name STRING,
    full_name STRING,
    department STRING,
    job_title STRING,
    warehouse_id STRING,
    hire_date DATE,
    status STRING,
    tenure_years DOUBLE,
    CONSTRAINT pk_employees PRIMARY KEY (employee_id)
)
USING DELTA;

In [0]:
%sql
INSERT INTO fashion_warehouse.gold.dim_employees
SELECT 
    employee_id,
    first_name,
    last_name,
    CONCAT(first_name, ' ', last_name) AS full_name,
    department,
    job_title,
    warehouse_id,
    hire_date,
    status,
    ROUND(DATEDIFF(CURRENT_DATE(), hire_date) / 365.25, 2) AS tenure_years
FROM fashion_warehouse.silver.employees;

In [0]:
%sql
SELECT * FROM fashion_warehouse.gold.dim_employees
LIMIT 5;

employee_id,first_name,last_name,full_name,department,job_title,warehouse_id,hire_date,status,tenure_years
1001,Joseph,Martin,Joseph Martin,Inventory,Inventory Controller,4,2023-02-23,Active,3.54
1002,Sarah,Evans,Sarah Evans,Inventory,Inventory Controller,20,2022-04-25,Active,4.37
1003,Amelia,Williams,Amelia Williams,Packing,Packer,13,2024-12-27,Active,1.69
1004,Michael,Wright,Michael Wright,Inventory,Inventory Controller,2,2024-10-03,Active,1.93
1005,David,Clarke,David Clarke,Dispatch,Dispatch Operative,11,2022-01-19,Active,4.63


In [0]:
%sql
CREATE OR REPLACE TABLE fashion_warehouse.gold.dim_suppliers
(
    supplier_name STRING,
    country STRING,
    supplier_tier STRING,
    lead_time_days DOUBLE,
    active_flag INT,
    CONSTRAINT pk_suppliers PRIMARY KEY (supplier_name)
)
USING DELTA;

In [0]:
%sql
INSERT INTO fashion_warehouse.gold.dim_suppliers
SELECT 
    supplier_name,
    country,
    supplier_tier,
    Lead_time_days,
    active_flag
FROM fashion_warehouse.silver.suppliers;

In [0]:
%sql
SELECT * FROM fashion_warehouse.gold.dim_suppliers
LIMIT 5; 

supplier_name,country,supplier_tier,lead_time_days,active_flag
Supplier 0001,India,B,20.0,1
Supplier 0002,India,B,29.0,1
Supplier 0003,Turkey,A,23.0,1
Supplier 0004,Turkey,B,8.0,1
Supplier 0005,Portugal,C,8.0,1


In [0]:
%sql
CREATE OR REPLACE TABLE fashion_warehouse.gold.dim_date
(
    date_key DATE,
    year INT,
    quarter INT,
    month INT,
    month_name STRING,
    week INT,
    day_of_month INT,
    day_of_week INT,
    day_name STRING,
    is_weekend BOOLEAN,
    is_month_start BOOLEAN,
    is_month_end BOOLEAN,
    is_quarter_start BOOLEAN,
    is_quarter_end BOOLEAN,
    is_year_start BOOLEAN,
    is_year_end BOOLEAN,
    CONSTRAINT pk_date PRIMARY KEY (date_key)
)
USING DELTA;

In [0]:
%sql
INSERT INTO fashion_warehouse.gold.dim_date
WITH date_range AS (
    SELECT EXPLODE(SEQUENCE(
        TO_DATE('2020-01-01'),
        TO_DATE('2030-12-31'),
        INTERVAL 1 DAY
    )) AS date_key
)
SELECT
    date_key,
    YEAR(date_key) AS year,
    QUARTER(date_key) AS quarter,
    MONTH(date_key) AS month,
    DATE_FORMAT(date_key, 'MMMM') AS month_name,
    WEEKOFYEAR(date_key) AS week,
    DAY(date_key) AS day_of_month,
    DAYOFWEEK(date_key) AS day_of_week,
    DATE_FORMAT(date_key, 'EEEE') AS day_name,
    CASE WHEN DAYOFWEEK(date_key) IN (1, 7) THEN TRUE ELSE FALSE END AS is_weekend,
    CASE WHEN DAY(date_key) = 1 THEN TRUE ELSE FALSE END AS is_month_start,
    CASE WHEN date_key = LAST_DAY(date_key) THEN TRUE ELSE FALSE END AS is_month_end,
    CASE WHEN MONTH(date_key) IN (1, 4, 7, 10) AND DAY(date_key) = 1 THEN TRUE ELSE FALSE END AS is_quarter_start,
    CASE WHEN MONTH(date_key) IN (3, 6, 9, 12) AND date_key = LAST_DAY(date_key) THEN TRUE ELSE FALSE END AS is_quarter_end,
    CASE WHEN MONTH(date_key) = 1 AND DAY(date_key) = 1 THEN TRUE ELSE FALSE END AS is_year_start,
    CASE WHEN MONTH(date_key) = 12 AND DAY(date_key) = 31 THEN TRUE ELSE FALSE END AS is_year_end
FROM date_range;

In [0]:
%sql
SELECT * FROM fashion_warehouse.gold.dim_date
LIMIT 5;

date_key,year,quarter,month,month_name,week,day_of_month,day_of_week,day_name,is_weekend,is_month_start,is_month_end,is_quarter_start,is_quarter_end,is_year_start,is_year_end
2020-01-01,2020,1,1,January,1,1,4,Wednesday,false,true,false,true,false,true,false
2020-01-02,2020,1,1,January,1,2,5,Thursday,false,false,false,false,false,false,false
2020-01-03,2020,1,1,January,1,3,6,Friday,false,false,false,false,false,false,false
2020-01-04,2020,1,1,January,1,4,7,Saturday,true,false,false,false,false,false,false
2020-01-05,2020,1,1,January,1,5,1,Sunday,true,false,false,false,false,false,false


In [0]:
%sql
CREATE OR REPLACE TABLE fashion_warehouse.gold.fact_orders
(
    order_id INT,
    order_date TIMESTAMP,
    order_date_key DATE,
    customer_id INT,
    warehouse_code STRING,
    sku STRING,
    quantity INT,
    sales_channel STRING,
    order_status STRING,
    unit_price DOUBLE,
    order_value DOUBLE,
    unit_cost DOUBLE,
    profit_amount DOUBLE,
    profit_margin_pct DOUBLE,
    
    CONSTRAINT pk_orders PRIMARY KEY (order_id),
    CONSTRAINT fk_orders_customer FOREIGN KEY (customer_id) REFERENCES fashion_warehouse.gold.dim_customers (customer_id),
    CONSTRAINT fk_orders_warehouse FOREIGN KEY (warehouse_code) REFERENCES fashion_warehouse.gold.dim_warehouses (warehouse_code),
    CONSTRAINT fk_orders_product FOREIGN KEY (sku) REFERENCES fashion_warehouse.gold.dim_products (sku),
    CONSTRAINT fk_orders_date FOREIGN KEY (order_date_key) REFERENCES fashion_warehouse.gold.dim_date (date_key)
)
USING DELTA;

In [0]:
%sql
INSERT INTO fashion_warehouse.gold.fact_orders
SELECT 
    o.order_id,
    o.order_date,
    CAST(o.order_date AS DATE) AS order_date_key,
    o.customer_id,
    o.warehouse_code,
    o.sku,
    o.quantity,
    o.sales_channel,
    o.order_status,
    o.unit_price,
    o.order_value,
    p.unit_cost,
    o.order_value - (o.quantity * p.unit_cost) AS profit_amount,
    CASE 
        WHEN o.order_value > 0 
        THEN ROUND(((o.order_value - (o.quantity * p.unit_cost)) / o.order_value) * 100, 2)
        ELSE 0 
    END AS profit_margin_pct
FROM fashion_warehouse.silver.orders o
LEFT JOIN fashion_warehouse.silver.products p
    ON o.sku = p.sku;

In [0]:
%sql
SELECT * FROM fashion_warehouse.gold.fact_orders
LIMIT 5;

order_id,order_date,order_date_key,customer_id,warehouse_code,sku,quantity,sales_channel,order_status,unit_price,order_value,unit_cost,profit_amount,profit_margin_pct
1,2025-10-03T16:08:32.000Z,2025-10-03,132555,WH12,SKU-000471,4,App,Completed,130.77,523.08,113.71,68.24000000000007,13.05
2,2025-06-16T14:39:16.000Z,2025-06-16,105360,WH03,SKU-002332,5,Store,Completed,233.84,1169.2,100.5,666.7,57.02
3,2025-03-09T21:34:01.000Z,2025-03-09,150392,WH15,SKU-001623,4,Store,Completed,69.22,276.88,60.19,36.120000000000005,13.05
4,2025-02-15T17:23:36.000Z,2025-02-15,146907,WH03,SKU-004783,4,Web,Completed,142.25,569.0,72.89,277.44,48.76
5,2026-02-23T04:24:58.000Z,2026-02-23,126838,WH02,SKU-004056,2,Web,Completed,84.31,168.62,73.31,22.0,13.05


In [0]:
%sql
CREATE OR REPLACE TABLE fashion_warehouse.gold.fact_deliveries
(
    delivery_id INT,
    supplier_name STRING,
    warehouse_code STRING,
    expected_date DATE,
    expected_date_key DATE,
    actual_date DATE,
    actual_date_key DATE,
    units_expected INT,
    units_received INT,
    units_diff INT,
    delivery_status STRING,
    delay_days INT,
    is_delayed BOOLEAN,
    is_partial BOOLEAN,
    is_over_received BOOLEAN,
    fill_rate_pct DOUBLE,
    
    CONSTRAINT pk_deliveries PRIMARY KEY (delivery_id),
    CONSTRAINT fk_deliveries_supplier FOREIGN KEY (supplier_name) REFERENCES fashion_warehouse.gold.dim_suppliers (supplier_name),
    CONSTRAINT fk_deliveries_warehouse FOREIGN KEY (warehouse_code) REFERENCES fashion_warehouse.gold.dim_warehouses (warehouse_code),
    CONSTRAINT fk_deliveries_expected_date FOREIGN KEY (expected_date_key) REFERENCES fashion_warehouse.gold.dim_date (date_key),
    CONSTRAINT fk_deliveries_actual_date FOREIGN KEY (actual_date_key) REFERENCES fashion_warehouse.gold.dim_date (date_key)
)
USING DELTA;

In [0]:
%sql
INSERT INTO fashion_warehouse.gold.fact_deliveries
SELECT 
    delivery_id,
    supplier_name,
    warehouse_code,
    expected_date,
    expected_date AS expected_date_key,
    actual_date,
    actual_date AS actual_date_key,
    units_expected,
    units_received,
    units_diff,
    delivery_status,
    delay_days,
    CASE WHEN delay_days > 0 THEN TRUE ELSE FALSE END AS is_delayed,
    CASE WHEN delivery_status = 'Partial Received' THEN TRUE ELSE FALSE END AS is_partial,
    CASE WHEN delivery_status = 'Over Received' THEN TRUE ELSE FALSE END AS is_over_received,
    CASE 
        WHEN units_expected > 0 
        THEN ROUND((units_received / units_expected) * 100, 2)
        ELSE 0 
    END AS fill_rate_pct
FROM fashion_warehouse.silver.deliveries;

In [0]:
%sql
SELECT * FROM fashion_warehouse.gold.fact_deliveries
LIMIT 5; 

delivery_id,supplier_name,warehouse_code,expected_date,expected_date_key,actual_date,actual_date_key,units_expected,units_received,units_diff,delivery_status,delay_days,is_delayed,is_partial,is_over_received,fill_rate_pct
27023,Supplier 0041,WH11,2026-09-04,2026-09-04,2026-09-10,2026-09-10,913,3930,3017,Over Received,6,true,false,true,430.45
31927,Supplier 0341,WH07,2026-09-04,2026-09-04,2026-09-10,2026-09-10,1953,4974,3021,Over Received,6,true,false,true,254.69
43488,Supplier 0109,WH20,2026-09-04,2026-09-04,2026-09-10,2026-09-10,652,1510,858,Over Received,6,true,false,true,231.6
2490,Supplier 0142,WH18,2026-09-04,2026-09-04,2026-09-09,2026-09-09,2294,1300,-994,Partial Received,5,true,true,false,56.67
48889,Supplier 0088,WH04,2026-09-04,2026-09-04,2026-09-09,2026-09-09,2397,601,-1796,Partial Received,5,true,true,false,25.07


In [0]:
%sql
CREATE OR REPLACE TABLE fashion_warehouse.gold.fact_stock_movements
(
    movement_id INT,
    timestamp TIMESTAMP,
    movement_date_key DATE,
    warehouse_code STRING,
    sku STRING,
    product_name STRING,
    employee_id INT,
    movement_type STRING,
    quantity INT,
    reference_id INT,
    
    CONSTRAINT pk_stock_movements PRIMARY KEY (movement_id),
    CONSTRAINT fk_stock_warehouse FOREIGN KEY (warehouse_code) REFERENCES fashion_warehouse.gold.dim_warehouses (warehouse_code),
    CONSTRAINT fk_stock_product FOREIGN KEY (sku) REFERENCES fashion_warehouse.gold.dim_products (sku),
    CONSTRAINT fk_stock_employee FOREIGN KEY (employee_id) REFERENCES fashion_warehouse.gold.dim_employees (employee_id),
    CONSTRAINT fk_stock_date FOREIGN KEY (movement_date_key) REFERENCES fashion_warehouse.gold.dim_date (date_key)
)
USING DELTA;

In [0]:
%sql
INSERT INTO fashion_warehouse.gold.fact_stock_movements
SELECT 
    movement_id,
    timestamp,
    CAST(timestamp AS DATE) AS movement_date_key,
    warehouse_code,
    sku,
    product_name,
    employee_id,
    movement_type,
    quantity,
    reference_id
FROM fashion_warehouse.silver.stock_movements;

In [0]:
%sql
SELECT * FROM fashion_warehouse.gold.fact_stock_movements
LIMIT 5;

movement_id,timestamp,movement_date_key,warehouse_code,sku,product_name,employee_id,movement_type,quantity,reference_id
1,2025-09-26T14:49:44.000Z,2025-09-26,WH12,SKU-002633,Puma Footwear,1616,ADJUSTMENT,97,62374
2,2025-10-08T17:18:12.000Z,2025-10-08,WH13,SKU-004020,Puma Accessories,1673,TRANSFER,29,71277
3,2025-07-07T22:47:58.000Z,2025-07-07,WH08,SKU-000814,Under Armour Accessories,2679,ADJUSTMENT,27,96692
4,2026-02-22T16:36:15.000Z,2026-02-22,WH04,SKU-002475,Reebok Womenswear,2957,ADJUSTMENT,35,2736
5,2025-02-07T06:44:15.000Z,2025-02-07,WH16,SKU-001763,Under Armour Accessories,1840,RETURN,37,36083


In [0]:
%sql
CREATE OR REPLACE TABLE fashion_warehouse.gold.fact_scanner_events
(
    event_id INT,
    timestamp TIMESTAMP,
    event_date_key DATE,
    scanner_id STRING,
    warehouse_code STRING,
    warehouse_name STRING,
    department STRING,
    action STRING,
    sku STRING,
    quantity INT,
    bin_location STRING,
    
    CONSTRAINT pk_scanner_events PRIMARY KEY (event_id),
    CONSTRAINT fk_scanner_warehouse FOREIGN KEY (warehouse_code) REFERENCES fashion_warehouse.gold.dim_warehouses (warehouse_code),
    CONSTRAINT fk_scanner_product FOREIGN KEY (sku) REFERENCES fashion_warehouse.gold.dim_products (sku),
    CONSTRAINT fk_scanner_date FOREIGN KEY (event_date_key) REFERENCES fashion_warehouse.gold.dim_date (date_key)
)
USING DELTA;

In [0]:
%sql
INSERT INTO fashion_warehouse.gold.fact_scanner_events
SELECT 
    event_id,
    timestamp,
    CAST(timestamp AS DATE) AS event_date_key,
    scanner_id,
    warehouse_code,
    warehouse_name,
    department,
    action,
    sku,
    quantity,
    bin_location
FROM fashion_warehouse.silver.scanner_events;

In [0]:
%sql
SELECT * FROM fashion_warehouse.gold.fact_scanner_events
LIMIT 5; 

event_id,timestamp,event_date_key,scanner_id,warehouse_code,warehouse_name,department,action,sku,quantity,bin_location
1,2025-07-04T16:47:54.000Z,2025-07-04,SC-018,WH20,Glasgow Distribution Centre,Packing,PACK,SKU-001592,45,D-42-02
2,2025-05-27T10:43:02.000Z,2025-05-27,SC-485,WH14,Nottingham Distribution Centre,Picking,PACK,SKU-000241,19,E-47-01
3,2026-07-06T08:37:09.000Z,2026-07-06,SC-071,WH03,Birmingham Distribution Centre,Packing,DISPATCH,SKU-002307,22,E-15-04
4,2025-11-01T07:02:31.000Z,2025-11-01,SC-213,WH18,Liverpool Distribution Centre,Returns,PICK,SKU-001767,4,G-07-02
5,2025-05-03T02:53:57.000Z,2025-05-03,SC-470,WH12,Manchester Distribution Centre,Packing,PICK,SKU-000760,43,G-53-05
